# Crawl TopCV từ sitemap

Hai bước: lấy link IT/CNTT/Data Science năm 2026 từ sitemap và lưu TXT, sau đó crawl chi tiết từ TXT và lưu JSONL. Chạy lại bước crawl để tiếp tục các tin chưa tải thành công.


## 1. Nạp crawler và xác định thư mục dự án

In [ ]:
import importlib
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'src' / 'crawlers' / 'topcv_it_jobs.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Hãy mở notebook từ thư mục dự án hoặc thư mục notebooks/.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.crawlers import topcv_it_jobs as crawler
crawler = importlib.reload(crawler)
print('Dự án:', PROJECT_ROOT)
print('JSONL:', crawler.RAW_JSONL)

## 2. Cấu hình

Chỉ cần chọn năm, số luồng và thời gian chờ. Đường dẫn lưu do crawler quản lý.
Chạy cell lấy link khi muốn quét mới; khi đã có file link thì bỏ qua cell đó và chạy cell crawl.


In [ ]:
YEAR = 2026  # Năm lastmod, không phải năm đăng tin
WORKERS = 2
DELAY = 2.0

print('File link:', crawler.job_urls_file(YEAR))
print('Dữ liệu thô:', crawler.RAW_JSONL)


## 3. Lấy link từ sitemap và lưu TXT

Chỉ giữ URL có từ khóa IT/CNTT/Data Science và lastmod thuộc YEAR. Chạy lại cell này để cập nhật file link.
Nếu đã có file link đã lọc thì bỏ qua cell này. Bước crawl bên dưới chỉ đọc file.

Slug có thể không nêu đầy đủ nghề nghiệp; bộ lọc chỉ giữ URL có từ khóa nhận diện được.


In [ ]:
job_urls = crawler.get_job_urls_from_sitemap(YEAR, delay=DELAY)
display(pd.DataFrame({'url': job_urls[:10]}))


## 4. Crawl chi tiết từ file link

Ô này chỉ đọc link đã lưu, không quét lại sitemap hay trang danh sách. Có thể chạy trong một phiên kernel mới sau các ô thiết lập/cấu hình. Khi dừng giữa chừng, các tin đã ghi vẫn nằm trong JSONL để lần sau tiếp tục.


In [ ]:
job_urls = crawler.load_job_urls(YEAR)
crawl_complete = crawler.crawl_from_links(job_urls, workers=WORKERS, delay=DELAY)
print('Lượt crawl hoàn tất:', crawl_complete)


## 5. Chuyển JSONL sang CSV và Parquet

Chạy sau cell nạp crawler; không cần chạy lại sitemap hoặc crawl.
Giữ các trường và các bản ghi trong JSONL. Nếu có dòng JSON lỗi do crawl bị ngắt, bỏ qua và báo số dòng lỗi.
Kết quả lưu ở `data/processed/topcv_jobs.csv` và `data/processed/topcv_jobs.parquet`; chạy lại cell sẽ ghi đè hai file này.


In [22]:
import json

records = []
invalid_lines = 0
with crawler.RAW_JSONL.open(encoding="utf-8-sig") as file:
    for line in file:
        if not line.strip():
            continue
        try:
            record = json.loads(line)
            if not isinstance(record, dict):
                raise ValueError("Bản ghi phải là object JSON")
            records.append(record)
        except ValueError:
            invalid_lines += 1

print(f"Đọc {len(records):,} bản ghi; bỏ qua {invalid_lines:,} dòng lỗi.")
if not records:
    raise ValueError("Không có bản ghi hợp lệ để xuất.")

jobs = pd.DataFrame(records)
output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
csv_file = output_dir / "topcv_jobs.csv"
parquet_file = output_dir / "topcv_jobs.parquet"

jobs.to_csv(csv_file, index=False, encoding="utf-8-sig")
jobs.to_parquet(parquet_file, index=False)

print("CSV:", csv_file)
print("Parquet:", parquet_file)
display(jobs.head())


Đọc 1,503 bản ghi; bỏ qua 0 dòng lỗi.
CSV: g:\hoc_tap\IntroDS\project_intro_ds\data\processed\topcv_jobs.csv
Parquet: g:\hoc_tap\IntroDS\project_intro_ds\data\processed\topcv_jobs.parquet


,job_id,url,list_title,list_company,list_salary,list_location,list_experience,crawled_at,title,company,...,experience_label,experience_months,education_label,position,job_description,requirements,benefits,skills,date_posted,valid_through
0,2309792,https://www.topcv.vn/viec-lam/nhan-vien-hanh-c...,Nhân Viên Hành Chính: Gán Nhãn Dữ Liệu Hình Ản...,CÔNG TY TNHH ALCHERA VIỆT NAM,6.8 - 10 triệu,Hồ Chí Minh (mới),Không yêu cầu,2026-10-05T03:15:00.764591+00:00,Nhân Viên Hành Chính: Gán Nhãn Dữ Liệu Hình Ản...,CÔNG TY TNHH ALCHERA VIỆT NAM,...,Không yêu cầu,NaN,Trung học phổ thông (Cấp 3) trở lên,Nhân viên,Thực hiện thao tác Data labeling (gán nhãn dữ ...,Không yêu cầu kinh nghiệm – được đào tạo từ đầ...,Thu nhập từ 6.800.000 - 10.000.000/ tháng. Thử...,"Nhanh nhẹn, Cẩn thận, Tỉ mỉ, Tin học văn phòng...",2026-09-22,2026-10-22T23:59:59+07:00
1,2309569,https://www.topcv.vn/viec-lam/fresher-web-deve...,Fresher Web Developer (GPA 3.2+) — Lương Khởi ...,Công ty Cổ phần MISA,Từ 15 triệu,Hà Nội,Không yêu cầu,2026-10-05T03:15:01.047488+00:00,Fresher Web Developer (GPA 3.2+) — Lương Khởi ...,Công ty Cổ phần MISA,...,Không yêu cầu,NaN,Đại Học trở lên,Thực tập sinh,Tham gia phát triển các sản phẩm và dịch vụ tà...,Học vấn: Tốt nghiệp cử nhân hoặc Kỹ sư các chu...,"Tiên phong công nghệ, uy tín Jetpay trực thuộc...","CSS, HTML, MySQL, OOP, SQL Server, Database, A...",2026-09-22,2026-10-22T23:59:59+07:00
2,2314545,https://www.topcv.vn/viec-lam/automation-teste...,Automation Tester - 2 Năm Kinh Nghiệm - Nghỉ T...,CÔNG TY TNHH GEMCOMMERCE,Tới 40 triệu,Hà Nội,2 năm,2026-10-05T03:15:01.090582+00:00,Automation Tester - 2 Năm Kinh Nghiệm - Nghỉ T...,CÔNG TY TNHH GEMCOMMERCE,...,2 năm,24.0,Đại Học trở lên,Nhân viên,"You’ll do Study and analyze requirements, syst...",You have... At least 2 years of experience in ...,We offer Salary range: Up to 40MIL Performance...,,2026-09-28,2026-10-28T23:59:59+07:00
3,2043180,https://www.topcv.vn/viec-lam/penetration-test...,Penetration Tester,Công Ty TNHH LG CNS VIỆT NAM,Thoả thuận,Hà Nội,1 năm,2026-10-05T03:15:01.129399+00:00,Penetration Tester,Công Ty TNHH LG CNS VIỆT NAM,...,1 năm,12.0,Đại Học trở lên,Nhân viên,Perform Web Application and API penetration te...,[Required] Bachelor’s degree in Computer Scien...,Attractive salary and bonus will be discussed ...,"Python, VPN, firewall, gỗ, IT Infrastructure, ...",2026-09-29,2026-10-29T23:59:59+07:00
4,2307663,https://www.topcv.vn/viec-lam/chuyen-vien-ha-t...,Chuyên Viên Hạ Tầng Công Nghệ Thông Tin,Tập Đoàn Tân Hoàng Minh,18 - 24 triệu,Hồ Chí Minh (mới) & Hà Nội,2 năm,2026-10-05T03:15:02.347211+00:00,Chuyên Viên Hạ Tầng Công Nghệ Thông Tin,Tập Đoàn Tân Hoàng Minh,...,2 năm,24.0,Trung cấp trở lên,Nhân viên,1. Hỗ trợ người dùng (IT Helpdesk / Support) T...,"Học vấn: Tốt nghiệp Trung cấp, Cao đẳng hoặc Đ...",Thu nhập: Thoả thuận theo năng lực Thưởng thán...,,2026-09-21,2026-10-21T23:59:59+07:00
